# Load and Compare Saved SAEs

This notebook loads the model first, then one or two saved `TrainableSAE` checkpoints, then runs a prompt through the base model and each SAE reconstruction.

In [ ]:
from pathlib import Path
import random
import sys
from datasets import load_dataset
import torch
print(torch.cuda.is_available())
print(torch.version.cuda)
# from datasets import load_dataset

REQUESTED_DEVICE = "cuda"  # Options: "auto", "cpu", "cuda", "cuda:0", "mps", ...


PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "trainable_sae.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"python = {sys.executable}")

from trainable_sae import (
    HookPointSpec,
    SAEConfig,
    SAEConnector,
    TrainableSAE,
    load_hooked_transformer,
    resolve_device,
)

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

device = resolve_device(REQUESTED_DEVICE)
dtype = torch.float32
device, dtype

True
12.6
PROJECT_ROOT = d:\andyh\Documents\Projects\mines\MATH498-Sparse-Autoencoder-Manipulation
python = d:\andyh\Documents\Projects\mines\MATH498-Sparse-Autoencoder-Manipulation\.venv\Scripts\python.exe


('cpu', torch.float32)

## Config

In [6]:
MODEL_NAME = "google/gemma-3-270m-it"

SAE_PATHS = [
    PROJECT_ROOT / "custom_saes/gemma3_270m_four_saes_1777260602/shrink",
    PROJECT_ROOT / "custom_saes/gemma3_270m_four_saes_1777260602/shrink",
]

EVAL_PROMPT = "tell me true or false: a square has five sides"
MAX_NEW_TOKENS = 20
TEMPERATURE = 0.0

## Load Model First

In [7]:
model = load_hooked_transformer(
    MODEL_NAME,
    device=device,
    dtype=dtype,
)

mid_layer = model.cfg.n_layers // 2
hook_point = HookPointSpec(layer=mid_layer, site="resid_post").name
d_in = model.cfg.d_model

print(f"model:      {MODEL_NAME}")
print(f"layers:     {model.cfg.n_layers}")
print(f"hook point: {hook_point}")
print(f"d_in:       {d_in}")

Loading weights: 100%|██████████| 236/236 [00:00<00:00, 3224.65it/s]


Loaded pretrained model google/gemma-3-270m-it into HookedTransformer
Moving model to device:  cpu
model:      google/gemma-3-270m-it
layers:     18
hook point: blocks.9.hook_resid_post
d_in:       640


## Load SAEs

In [8]:
def load_sae_connector(path: Path):
    path = Path(path)
    sae = TrainableSAE.load(path, device=device)
    hook_point = sae.cfg.metadata.get("hook_point")
    if hook_point is None:
        raise ValueError(f"{path} does not include hook_point metadata.")

    if sae.cfg.d_in != model.cfg.d_model:
        raise ValueError(f"SAE d_in={sae.cfg.d_in}, model d_model={model.cfg.d_model}")

    connector = SAEConnector(model=model, sae=sae, hook_point=hook_point, device=device)
    label = sae.cfg.metadata.get("variant", path.name)
    return {
        "label": label,
        "path": path,
        "sae": sae,
        "connector": connector,
        "hook_point": hook_point,
    }


loaded_saes = [load_sae_connector(path) for path in SAE_PATHS]

for item in loaded_saes:
    cfg = item["sae"].cfg
    print(
        f"{item['label']}: {cfg.d_in} -> {cfg.d_sae}, "
        f"activation={cfg.activation}, hook={item['hook_point']}"
    )

Moving model to device:  cpu
Moving model to device:  cpu
shrink: 640 -> 10240, activation=shrink, hook=blocks.9.hook_resid_post
shrink: 640 -> 10240, activation=shrink, hook=blocks.9.hook_resid_post


## Prompt Comparison

In [9]:
generate_kwargs = dict(
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    do_sample=bool(TEMPERATURE > 0),
    prepend_bos=False,
)


def clean_generation(text: str) -> str:
    return text.replace("<bos>", "").replace("<eos>", "").strip()


tokens = model.to_tokens(EVAL_PROMPT).to(device)
prompt_len = tokens.shape[1]

with torch.no_grad():
    normal_tokens = model.generate(tokens, **generate_kwargs)
normal_output = clean_generation(model.to_string(normal_tokens[0, prompt_len:]))

print("Prompt:")
print(EVAL_PROMPT)
print("\nBase model:")
print(normal_output)

for item in loaded_saes:
    with torch.no_grad():
        sae_output = clean_generation(
            item["connector"].generate_with_sae(EVAL_PROMPT, mode="reconstruct", **generate_kwargs)
        )
    print(f"\nWith SAE ({item['label']}):")
    print(sae_output)

100%|██████████| 20/20 [00:07<00:00,  2.84it/s]


Prompt:
tell me true or false: a square has five sides

Base model:
.
A square has 4 sides.
A square has 4 sides.
A square


100%|██████████| 20/20 [00:07<00:00,  2.80it/s]



With SAE (shrink):
, it is a physical entity, it are a threat against the world, it are a threat against


100%|██████████| 20/20 [00:06<00:00,  2.91it/s]


With SAE (shrink):
, it is a physical entity, it are a threat against the world, it are a threat against


## Activation Snapshot

In [10]:
def count_nonzero_features(item, prompt: str, min_abs: float = 0.0):
    connector = item["connector"]
    tokens = model.to_tokens(prompt).to(device)
    acts = connector.collect_activations(tokens).to(dtype=torch.float32)
    with torch.no_grad():
        features = item["sae"].encode(acts)

    scores = features[0].abs().max(dim=0).values
    return int((scores > min_abs).sum().item())


for item in loaded_saes:
    count = count_nonzero_features(item, EVAL_PROMPT)
    print(f"{item['label']}: {count} non-zero features")

shrink: 1299 non-zero features
shrink: 1299 non-zero features
